In [46]:
import mne
import handy.my_utils as deniz
import warnings

warnings.filterwarnings("ignore")

In [47]:
dataset_path = 'data-understanding/data/chb-mit'
subject_folders = deniz.get_folder_names(dataset_path)
edf_paths = deniz.get_edf_paths(dataset_path, subject_folders)

# Channel Processing Workflow
- **Iterate through all EDF files:** We process every EDF file in the dataset.
- **Store channels in a dictionary:** All channels from each file are recorded into a dictionary for centralized access.
- **Minimal channel cleaning:**
  - Remove hidden/irrelevant annotations or artifacts.
  - Trim leading/trailing gaps or incomplete segments.
- **Target channel validation:** Based on our research, we check if the predefined **target channels** (critical for analysis) exist in every file.
- **Result:** The check returns **True**—all target channels are present across the dataset.



In [48]:
channels_dict = deniz.get_channels_as_dict(dataset_path, subject_folders, json_name='channels', save=False)
cleaned_dict = deniz.normalize_channel_dict(channels_dict)
channels = [item for sublist in cleaned_dict.values() for item in sublist]

In [49]:
TARGET_CHANNELS = [
    'FP1-F7', 'F7-T7', 'T7-P7', 'P7-O1',
    'FP1-F3', 'F3-C3', 'C3-P3', 'P3-O1',
    'FP2-F4', 'F4-C4', 'C4-P4', 'P4-O2',
    'FP2-F8', 'F8-T8', 'T8-P8', 'P8-O2',
    'FZ-CZ', 'CZ-PZ'
]

print(set(TARGET_CHANNELS) <= set(channels))

True


Check the dict we cleaned.

In [50]:
for key, old_list in channels_dict.items():
    new_list = cleaned_dict.get(key, [])

    # Küme farkı alarak eski listede olup yeni listede olmayanları buluyoruz
    removed = set(old_list) - set(new_list)
    added = set(new_list) - set(old_list)

    if removed or added:
        print(f"--- Klasör: {key} ---")
        if removed: print(f"Silinen/Değişen: {removed}")
        if added:   print(f"Yeni Eklenen/Düzeltilen: {added}")

--- Klasör: data/chb-mit\chb15\chb15_01.edf ---
Silinen/Değişen: {'FC2-Ref', 'FC6-Ref', 'CP1-Ref', 'CP6-Ref', 'CP2-Ref', 'FC1-Ref', 'FC5-Ref', 'CP5-Ref'}
Yeni Eklenen/Düzeltilen: {'CP2-REF', 'CP5-REF', 'FC6-REF', 'FC2-REF', 'CP6-REF', 'CP1-REF', 'FC1-REF', 'FC5-REF'}
--- Klasör: data/chb-mit\chb15\chb15_02.edf ---
Silinen/Değişen: {'FC2-Ref', 'FC6-Ref', 'CP1-Ref', 'CP6-Ref', 'CP2-Ref', 'FC1-Ref', 'FC5-Ref', 'CP5-Ref'}
Yeni Eklenen/Düzeltilen: {'CP2-REF', 'CP5-REF', 'FC6-REF', 'FC2-REF', 'CP6-REF', 'CP1-REF', 'FC1-REF', 'FC5-REF'}
--- Klasör: data/chb-mit\chb15\chb15_03.edf ---
Silinen/Değişen: {'FC2-Ref', 'FC6-Ref', 'CP1-Ref', 'CP6-Ref', 'CP2-Ref', 'FC1-Ref', 'FC5-Ref', 'CP5-Ref'}
Yeni Eklenen/Düzeltilen: {'CP2-REF', 'CP5-REF', 'FC6-REF', 'FC2-REF', 'CP6-REF', 'CP1-REF', 'FC1-REF', 'FC5-REF'}
--- Klasör: data/chb-mit\chb15\chb15_04.edf ---
Silinen/Değişen: {'FC2-Ref', 'FC6-Ref', 'CP1-Ref', 'CP6-Ref', 'CP2-Ref', 'FC1-Ref', 'FC5-Ref', 'CP5-Ref'}
Yeni Eklenen/Düzeltilen: {'CP2-REF', 'CP5-

# Channel Inspection for Each Patient
* From our preliminary research, we know that some files contain variants of the **T8-P8** channel, such as **T8-P8-0** and **T8-P8-1**, while others only have **T8-P8**.
* To avoid losing this data, we dynamically update the **target_channel** list to include all relevant variants.
```---


In [51]:
TARGET_CHANNELS = [
    'FP1-F7', 'F7-T7', 'T7-P7', 'P7-O1',
    'FP1-F3', 'F3-C3', 'C3-P3', 'P3-O1',
    'FP2-F4', 'F4-C4', 'C4-P4', 'P4-O2',
    'FP2-F8', 'F8-T8', 'T8-P8-0', 'T8-P8-1', 'P8-O2', 'T8-P8',
    'FZ-CZ', 'CZ-PZ'
]

# Checking Each File Individually
* We verify if there is any discrepancy in our **target channels**—i.e., whether any expected channel is missing or if there are unexpected entries.
* To optimize search speed, we convert the **target_channel** list into a **set** for faster lookups.


In [52]:
target_set = set(TARGET_CHANNELS)
missing_files = []
dataset_path = 'data-understanding/data/chb-mit'
subject_folders = deniz.get_folder_names(dataset_path)
CHB01_paths = deniz.get_edf_paths(dataset_path, subject_folders[0])
CHB02_paths = deniz.get_edf_paths(dataset_path, subject_folders[1])
CHB03_paths = deniz.get_edf_paths(dataset_path, subject_folders[2])
CHB04_paths = deniz.get_edf_paths(dataset_path, subject_folders[3])
CHB05_paths = deniz.get_edf_paths(dataset_path, subject_folders[4])
CHB06_paths = deniz.get_edf_paths(dataset_path, subject_folders[5])
CHB07_paths = deniz.get_edf_paths(dataset_path, subject_folders[6])
CHB08_paths = deniz.get_edf_paths(dataset_path, subject_folders[7])
CHB09_paths = deniz.get_edf_paths(dataset_path, subject_folders[8])
CHB10_paths = deniz.get_edf_paths(dataset_path, subject_folders[9])
CHB11_paths = deniz.get_edf_paths(dataset_path, subject_folders[10])
CHB12_paths = deniz.get_edf_paths(dataset_path, subject_folders[11])
CHB13_paths = deniz.get_edf_paths(dataset_path, subject_folders[12])
CHB14_paths = deniz.get_edf_paths(dataset_path, subject_folders[13])
CHB15_paths = deniz.get_edf_paths(dataset_path, subject_folders[14])
CHB16_paths = deniz.get_edf_paths(dataset_path, subject_folders[15])
CHB17_paths = deniz.get_edf_paths(dataset_path, subject_folders[16])
CHB18_paths = deniz.get_edf_paths(dataset_path, subject_folders[17])
CHB19_paths = deniz.get_edf_paths(dataset_path, subject_folders[18])
CHB20_paths = deniz.get_edf_paths(dataset_path, subject_folders[19])
CHB21_paths = deniz.get_edf_paths(dataset_path, subject_folders[20])
CHB22_paths = deniz.get_edf_paths(dataset_path, subject_folders[21])
CHB23_paths = deniz.get_edf_paths(dataset_path, subject_folders[22])
CHB24_paths = deniz.get_edf_paths(dataset_path, subject_folders[23])

# CHB01
* T8-P8 is missing in all files.

In [53]:
deniz.check_is_every_edf_containts_target_chs(edf_paths=CHB01_paths, target_set=target_set, missing_files=missing_files)

❌ Eksik kanal var: chb-mit\chb01\chb01_43.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb01\chb01_46.edf -> Eksikler: {'T8-P8'}


# CHB02
* T8-P8 is missing in all files.

In [54]:
deniz.check_is_every_edf_containts_target_chs(edf_paths=CHB02_paths, target_set=target_set, missing_files=missing_files)


❌ Eksik kanal var: chb-mit\chb02\chb02_33.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb02\chb02_34.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb02\chb02_35.edf -> Eksikler: {'T8-P8'}


# CHB03
* T8-P8 is missing in all files.

In [55]:
deniz.check_is_every_edf_containts_target_chs(edf_paths=CHB03_paths, target_set=target_set, missing_files=missing_files)

❌ Eksik kanal var: chb-mit\chb03\chb03_30.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb03\chb03_31.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb03\chb03_32.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb03\chb03_33.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb03\chb03_34.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb03\chb03_35.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb03\chb03_36.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb03\chb03_37.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb03\chb03_38.edf -> Eksikler: {'T8-P8'}


# CHB04
* T8-P8 is missing in all files.

In [56]:
deniz.check_is_every_edf_containts_target_chs(edf_paths=CHB04_paths, target_set=target_set, missing_files=missing_files)


❌ Eksik kanal var: chb-mit\chb04\chb04_27.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb04\chb04_28.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb04\chb04_29.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb04\chb04_30.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb04\chb04_31.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb04\chb04_32.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb04\chb04_33.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb04\chb04_34.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb04\chb04_35.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb04\chb04_36.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb04\chb04_37.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb04\chb04_38.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb04\chb04_39.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb04\chb04_40.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb04\c

# CHB05
* T8-P8 is missing in all files.

In [57]:
deniz.check_is_every_edf_containts_target_chs(edf_paths=CHB05_paths, target_set=target_set, missing_files=missing_files)



❌ Eksik kanal var: chb-mit\chb05\chb05_39.edf -> Eksikler: {'T8-P8'}


# CHB06
* T8-P8 is missing in all files.

In [58]:
deniz.check_is_every_edf_containts_target_chs(edf_paths=CHB06_paths, target_set=target_set, missing_files=missing_files)


❌ Eksik kanal var: chb-mit\chb06\chb06_24.edf -> Eksikler: {'T8-P8'}


# CHB07
* T8-P8 is missing in all files.

In [59]:
deniz.check_is_every_edf_containts_target_chs(edf_paths=CHB07_paths, target_set=target_set, missing_files=missing_files)


❌ Eksik kanal var: chb-mit\chb07\chb07_05.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb07\chb07_06.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb07\chb07_07.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb07\chb07_08.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb07\chb07_09.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb07\chb07_10.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb07\chb07_11.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb07\chb07_12.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb07\chb07_13.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb07\chb07_14.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb07\chb07_15.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb07\chb07_16.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb07\chb07_17.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb07\chb07_18.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb07\c

# CHB08
* T8-P8 is missing in all files.

In [60]:
deniz.check_is_every_edf_containts_target_chs(edf_paths=CHB08_paths, target_set=target_set, missing_files=missing_files)


❌ Eksik kanal var: chb-mit\chb08\chb08_10.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb08\chb08_11.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb08\chb08_12.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb08\chb08_13.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb08\chb08_14.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb08\chb08_15.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb08\chb08_16.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb08\chb08_17.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb08\chb08_18.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb08\chb08_19.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb08\chb08_20.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb08\chb08_21.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb08\chb08_22.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb08\chb08_23.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb08\c

# CHB09
* T8-P8 is missing in all files.

In [61]:
deniz.check_is_every_edf_containts_target_chs(edf_paths=CHB09_paths, target_set=target_set, missing_files=missing_files)


❌ Eksik kanal var: chb-mit\chb09\chb09_01.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb09\chb09_02.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb09\chb09_03.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb09\chb09_04.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb09\chb09_05.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb09\chb09_06.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb09\chb09_07.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb09\chb09_08.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb09\chb09_09.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb09\chb09_10.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb09\chb09_11.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb09\chb09_12.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb09\chb09_13.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb09\chb09_14.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb09\c

# CHB10
* T8-P8 is missing in all files.

In [62]:
deniz.check_is_every_edf_containts_target_chs(edf_paths=CHB10_paths, target_set=target_set, missing_files=missing_files)


❌ Eksik kanal var: chb-mit\chb10\chb10_01.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb10\chb10_02.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb10\chb10_03.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb10\chb10_04.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb10\chb10_05.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb10\chb10_06.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb10\chb10_07.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb10\chb10_08.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb10\chb10_12.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb10\chb10_13.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb10\chb10_14.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb10\chb10_15.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb10\chb10_16.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb10\chb10_17.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb10\c

# CHB11
* T8-P8 is missing in all files.

In [63]:
deniz.check_is_every_edf_containts_target_chs(edf_paths=CHB11_paths, target_set=target_set, missing_files=missing_files)


❌ Eksik kanal var: chb-mit\chb11\chb11_60.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb11\chb11_61.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb11\chb11_62.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb11\chb11_63.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb11\chb11_82.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb11\chb11_92.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb11\chb11_99.edf -> Eksikler: {'T8-P8'}


# CHB13
* In a few cases, T8-P8 is missing, and in a few others, T8-P8-0 and T8-P8-1 are missing.

In [64]:
deniz.check_is_every_edf_containts_target_chs(edf_paths=CHB13_paths, target_set=target_set, missing_files=missing_files)


❌ Eksik kanal var: chb-mit\chb13\chb13_21.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb13\chb13_22.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb13\chb13_24.edf -> Eksikler: {'T8-P8-1', 'T8-P8-0'}
❌ Eksik kanal var: chb-mit\chb13\chb13_30.edf -> Eksikler: {'T8-P8-1', 'T8-P8-0'}
❌ Eksik kanal var: chb-mit\chb13\chb13_36.edf -> Eksikler: {'T8-P8-1', 'T8-P8-0'}
❌ Eksik kanal var: chb-mit\chb13\chb13_37.edf -> Eksikler: {'T8-P8-1', 'T8-P8-0'}
❌ Eksik kanal var: chb-mit\chb13\chb13_38.edf -> Eksikler: {'T8-P8-1', 'T8-P8-0'}
❌ Eksik kanal var: chb-mit\chb13\chb13_39.edf -> Eksikler: {'T8-P8-1', 'T8-P8-0'}
❌ Eksik kanal var: chb-mit\chb13\chb13_40.edf -> Eksikler: {'T8-P8-1', 'T8-P8-0'}
❌ Eksik kanal var: chb-mit\chb13\chb13_47.edf -> Eksikler: {'T8-P8-1', 'T8-P8-0'}
❌ Eksik kanal var: chb-mit\chb13\chb13_55.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb13\chb13_56.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb13\chb13_58.edf -> Eksikler: {'T8-P8'}

# CHB14
* T8-P8 is missing in all files.

In [65]:
deniz.check_is_every_edf_containts_target_chs(edf_paths=CHB14_paths, target_set=target_set, missing_files=missing_files)


❌ Eksik kanal var: chb-mit\chb14\chb14_07.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb14\chb14_11.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb14\chb14_12.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb14\chb14_13.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb14\chb14_14.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb14\chb14_16.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb14\chb14_17.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb14\chb14_18.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb14\chb14_19.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb14\chb14_20.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb14\chb14_22.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb14\chb14_24.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb14\chb14_25.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb14\chb14_26.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb14\c

# CHB15
* In only **one case**, the channels **{'T8-P8-0', 'T8-P8-1'}** are present, while in the remaining cases, **'T8-P8'** is missing.

In [66]:
deniz.check_is_every_edf_containts_target_chs(edf_paths=CHB15_paths, target_set=target_set, missing_files=missing_files)

❌ Eksik kanal var: chb-mit\chb15\chb15_32.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb15\chb15_33.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb15\chb15_35.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb15\chb15_37.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb15\chb15_40.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb15\chb15_45.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb15\chb15_46.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb15\chb15_49.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb15\chb15_50.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb15\chb15_51.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb15\chb15_52.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb15\chb15_54.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb15\chb15_61.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb15\chb15_62.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb15\c

# CHB16
* In the **last two cases**, the channels **{'T8-P8-0', 'T8-P8-1'}** are present, while in the **remaining cases**, only **{'T8-P8'}** exists.

In [67]:
deniz.check_is_every_edf_containts_target_chs(edf_paths=CHB16_paths, target_set=target_set, missing_files=missing_files)

❌ Eksik kanal var: chb-mit\chb16\chb16_08.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb16\chb16_09.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb16\chb16_10.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb16\chb16_11.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb16\chb16_12.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb16\chb16_13.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb16\chb16_14.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb16\chb16_15.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb16\chb16_16.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb16\chb16_17.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb16\chb16_18.edf -> Eksikler: {'T8-P8-1', 'T8-P8-0'}
❌ Eksik kanal var: chb-mit\chb16\chb16_19.edf -> Eksikler: {'T8-P8-1', 'T8-P8-0'}


# CHB17
* In the **last case**, the channels **{'T8-P8-0', 'T8-P8-1'}** are present, while in the **remaining cases**, only **{'T8-P8'}** is missing.

In [68]:
deniz.check_is_every_edf_containts_target_chs(edf_paths=CHB17_paths, target_set=target_set, missing_files=missing_files)

❌ Eksik kanal var: chb-mit\chb17\chb17c_13.edf -> Eksikler: {'T8-P8-1', 'T8-P8-0'}


# CHB18
* In the **first case**, the channels **{'T8-P8-0', 'T8-P8-1'}** are present, while in the **remaining cases**, only **{'T8-P8'}** is missing.

In [69]:
deniz.check_is_every_edf_containts_target_chs(edf_paths=CHB18_paths, target_set=target_set, missing_files=missing_files)


❌ Eksik kanal var: chb-mit\chb18\chb18_31.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb18\chb18_32.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb18\chb18_33.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb18\chb18_34.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb18\chb18_35.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb18\chb18_36.edf -> Eksikler: {'T8-P8'}


# CHB19
* In the **first case**, the channels **{'T8-P8-0', 'T8-P8-1'}** are present, while in the **remaining cases**, only **{'T8-P8'}** is missing.

In [70]:
deniz.check_is_every_edf_containts_target_chs(edf_paths=CHB19_paths, target_set=target_set, missing_files=missing_files)


❌ Eksik kanal var: chb-mit\chb19\chb19_06.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb19\chb19_07.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb19\chb19_08.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb19\chb19_09.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb19\chb19_10.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb19\chb19_11.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb19\chb19_12.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb19\chb19_13.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb19\chb19_14.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb19\chb19_15.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb19\chb19_16.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb19\chb19_17.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb19\chb19_18.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb19\chb19_19.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb19\c

# CHB20
* In all of the channels only **{'T8-P8'}** does not exist.

In [71]:
deniz.check_is_every_edf_containts_target_chs(edf_paths=CHB20_paths, target_set=target_set, missing_files=missing_files)


❌ Eksik kanal var: chb-mit\chb20\chb20_02.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb20\chb20_03.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb20\chb20_04.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb20\chb20_05.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb20\chb20_06.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb20\chb20_07.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb20\chb20_08.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb20\chb20_11.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb20\chb20_12.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb20\chb20_13.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb20\chb20_14.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb20\chb20_15.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb20\chb20_16.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb20\chb20_17.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb20\c

# CHB21
* In all of the channels only **{'T8-P8'}** does not exist.

In [72]:
deniz.check_is_every_edf_containts_target_chs(edf_paths=CHB21_paths, target_set=target_set, missing_files=missing_files)


❌ Eksik kanal var: chb-mit\chb21\chb21_33.edf -> Eksikler: {'T8-P8'}


# CHB22
* In all of the channels only **{'T8-P8'}** does not exist.

In [73]:
deniz.check_is_every_edf_containts_target_chs(edf_paths=CHB22_paths, target_set=target_set, missing_files=missing_files)


❌ Eksik kanal var: chb-mit\chb22\chb22_29.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb22\chb22_30.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb22\chb22_38.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb22\chb22_51.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb22\chb22_54.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb22\chb22_77.edf -> Eksikler: {'T8-P8'}


In [74]:
deniz.check_is_every_edf_containts_target_chs(edf_paths=CHB22_paths, target_set=target_set, missing_files=missing_files)


❌ Eksik kanal var: chb-mit\chb22\chb22_15.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb22\chb22_16.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb22\chb22_17.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb22\chb22_18.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb22\chb22_19.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb22\chb22_20.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb22\chb22_21.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb22\chb22_22.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb22\chb22_23.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb22\chb22_24.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb22\chb22_25.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb22\chb22_26.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb22\chb22_27.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb22\chb22_28.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb22\c

# CHB23
* In all of the channels only **{'T8-P8'}** does not exist.

In [75]:
deniz.check_is_every_edf_containts_target_chs(edf_paths=CHB23_paths, target_set=target_set, missing_files=missing_files)


❌ Eksik kanal var: chb-mit\chb23\chb23_06.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb23\chb23_07.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb23\chb23_08.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb23\chb23_09.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb23\chb23_10.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb23\chb23_16.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb23\chb23_17.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb23\chb23_19.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb23\chb23_20.edf -> Eksikler: {'T8-P8'}


# CHB24
* In all of the channels only **{'T8-P8'}** does not exist.

In [76]:
deniz.check_is_every_edf_containts_target_chs(edf_paths=CHB24_paths, target_set=target_set, missing_files=missing_files)


❌ Eksik kanal var: chb-mit\chb24\chb24_07.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb24\chb24_08.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb24\chb24_09.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb24\chb24_10.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb24\chb24_11.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb24\chb24_12.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb24\chb24_13.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb24\chb24_14.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb24\chb24_15.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb24\chb24_16.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb24\chb24_17.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb24\chb24_18.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb24\chb24_19.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb24\chb24_20.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb24\c

# CHB12
* Many missing channels...

In [77]:
deniz.check_is_every_edf_containts_target_chs(edf_paths=CHB12_paths, target_set=target_set, missing_files=missing_files)

❌ Eksik kanal var: chb-mit\chb12\chb12_37.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb12\chb12_38.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb12\chb12_39.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb12\chb12_40.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb12\chb12_41.edf -> Eksikler: {'T8-P8'}
❌ Eksik kanal var: chb-mit\chb12\chb12_42.edf -> Eksikler: {'T8-P8'}


In [78]:
# To controll through ch_names this cell could be used
dataset_path = 'data-understanding/data/chb-mit'
subject_folders = deniz.get_folder_names(dataset_path)
edf_paths = deniz.get_edf_paths(dataset_path, subject_folders[0])

raw = mne.io.read_raw_edf(CHB12_paths[11], preload=False)

print(raw.ch_names)

['F7', 'T7', 'P7', '--0', 'FP1', 'F3', 'C3', 'P3', '01', '--1', 'FZ', 'CZ', 'PZ', '--2', 'FP2', 'F4', 'C4', 'P4', 'O2', '--3', 'F8', 'T8', 'P8', 'EKG1-CHIN', 'C2', 'C6', 'CP2', 'CP4', 'CP6']


---
**Channel Standardization and Data Selection**

Based on our research, it appears that **all the channels we identified as critical are present in every EDF file**. However, we need to address inconsistencies in the naming of the **T8-P8** channel, which sometimes appears as **T8-P8-1** or **T8-P8-0**.

**Action Plan:**
- **Standardize channel names:** Convert either **T8-P8-1** or **T8-P8-0** to **T8-P8** to ensure uniformity across the dataset.
- **Drop redundant channels:** Remove any remaining non-standard channels to maintain a consistent number of channels in every file.

---

---

**Data Selection for Seizure Detection**
We have decided to **focus exclusively on files containing seizure events**. Here’s why:
- Each file spans **3600 seconds**, but not all files contain seizure activity.
- Even if we only retain seizure-containing files, the data remains **highly imbalanced**:
  - **~3% seizure duration**
  - **~97% non-seizure duration**
- Given that EEG data is inherently a **time-series**, balancing the dataset by dropping non-seizure segments is **not straightforward** and may lead to unintended loss of critical temporal information.

---

---

**Processing Seizure-Containing EDF Files**
We will:
- Parse the **summary.txt** file for each patient to identify EDF files containing seizure events.
- Build a **DataFrame** to organize and store the relevant information.
- Copy these seizure-containing EDF files into a **new directory** for further analysis.

---

In [79]:
df = deniz.build_seizure_dataframe(dataset_path)
df['path'] = df['folder'] + '/' + df['file']
df.to_csv('only_seizure_dataframe.csv')

In [80]:
df.query("file == 'chb12_29.edf'")
df.query("file == 'chb12_28.edf'")
df.query("file == 'chb12_27.edf'")

,folder,file,seizure_start_sec,seizure_end_sec,path
72,chb12,chb12_27.edf,916,951,chb12/chb12_27.edf
73,chb12,chb12_27.edf,1097,1124,chb12/chb12_27.edf
74,chb12,chb12_27.edf,1728,1753,chb12/chb12_27.edf
75,chb12,chb12_27.edf,1921,1963,chb12/chb12_27.edf
76,chb12,chb12_27.edf,2388,2440,chb12/chb12_27.edf
77,chb12,chb12_27.edf,2621,2669,chb12/chb12_27.edf


### Dropping Files from chb12
The seizure-containing files in the **chb12** dataset have **significant channel deficiencies**. Due to this inconsistency, we are **dropping these files** from our analysis to maintain data integrity and ensure uniformity across the dataset.


In [81]:
seizured_file_list = []
for i in df['path']:
    seizured_file_list.append(i)

to_remove = {
    'data/chb-mit/chb12/chb12_29.edf',
    'data/chb-mit/chb12/chb12_27.edf',
    'data/chb-mit/chb12/chb12_28.edf'
}
seizured_file_list = set(seizured_file_list)
seizured_file_list.difference_update(to_remove)
# control
print("chb12_29.edf" in seizured_file_list)
seizured_file_list = list(seizured_file_list)
seizured_file_list = sorted(seizured_file_list)
seizured_file_list = [dataset_path + '/' + item for item in seizured_file_list]

False


In [82]:
seizured_file_list

['data/chb-mit/chb01/chb01_03.edf',
 'data/chb-mit/chb01/chb01_04.edf',
 'data/chb-mit/chb01/chb01_15.edf',
 'data/chb-mit/chb01/chb01_16.edf',
 'data/chb-mit/chb01/chb01_18.edf',
 'data/chb-mit/chb01/chb01_21.edf',
 'data/chb-mit/chb01/chb01_26.edf',
 'data/chb-mit/chb02/chb02_16+.edf',
 'data/chb-mit/chb02/chb02_16.edf',
 'data/chb-mit/chb02/chb02_19.edf',
 'data/chb-mit/chb03/chb03_01.edf',
 'data/chb-mit/chb03/chb03_02.edf',
 'data/chb-mit/chb03/chb03_03.edf',
 'data/chb-mit/chb03/chb03_04.edf',
 'data/chb-mit/chb03/chb03_34.edf',
 'data/chb-mit/chb03/chb03_35.edf',
 'data/chb-mit/chb03/chb03_36.edf',
 'data/chb-mit/chb04/chb04_05.edf',
 'data/chb-mit/chb04/chb04_08.edf',
 'data/chb-mit/chb04/chb04_28.edf',
 'data/chb-mit/chb05/chb05_06.edf',
 'data/chb-mit/chb05/chb05_13.edf',
 'data/chb-mit/chb05/chb05_16.edf',
 'data/chb-mit/chb05/chb05_17.edf',
 'data/chb-mit/chb05/chb05_22.edf',
 'data/chb-mit/chb06/chb06_01.edf',
 'data/chb-mit/chb06/chb06_04.edf',
 'data/chb-mit/chb06/chb06_

In [83]:
df['path']

0      chb01/chb01_03.edf
1      chb01/chb01_04.edf
2      chb01/chb01_15.edf
3      chb01/chb01_16.edf
4      chb01/chb01_18.edf
              ...        
193    chb24/chb24_13.edf
194    chb24/chb24_14.edf
195    chb24/chb24_15.edf
196    chb24/chb24_17.edf
197    chb24/chb24_21.edf
Name: path, Length: 198, dtype: object

In [84]:
df

,folder,file,seizure_start_sec,seizure_end_sec,path
0,chb01,chb01_03.edf,2996,3036,chb01/chb01_03.edf
1,chb01,chb01_04.edf,1467,1494,chb01/chb01_04.edf
2,chb01,chb01_15.edf,1732,1772,chb01/chb01_15.edf
3,chb01,chb01_16.edf,1015,1066,chb01/chb01_16.edf
4,chb01,chb01_18.edf,1720,1810,chb01/chb01_18.edf
...,...,...,...,...,...
193,chb24,chb24_13.edf,3288,3304,chb24/chb24_13.edf
194,chb24,chb24_14.edf,1939,1966,chb24/chb24_14.edf
195,chb24,chb24_15.edf,3552,3569,chb24/chb24_15.edf
196,chb24,chb24_17.edf,3515,3581,chb24/chb24_17.edf


In [85]:
exclude_list = [
    'chb12/chb12_29.edf',
    'chb12/chb12_27.edf',
    'chb12/chb12_28.edf'
]
filtered_df = df[~df['path'].isin(exclude_list)]
total_seizure_elapse = (filtered_df['seizure_end_sec'] - filtered_df['seizure_start_sec']).sum()
print(f"Total Seizure Duration: {total_seizure_elapse}")
total_time = len(seizured_file_list) * 3600
percent = (total_seizure_elapse / total_time) * 100
print('Total seizure percentage in all edf files: %', percent)

Total Seizure Duration: 11125
Total seizure percentage in all edf files: % 2.1916863672182822


In [86]:
deniz.check_is_every_edf_containts_target_chs(seizured_file_list, target_set=target_set, missing_files=missing_files)

❌ Eksik kanal var: chb24_21.edf -> Eksikler: {'T8-P8'}


In [87]:
# deniz.copy_files_to_seizured_folder(seizured_file_list)

### Only seizures for preprocess.csv file creation

In [92]:
seizured_prep_list= "data/chb-mit"

df= deniz.build_seizure_dataframe(seizured_prep_list)
exclude_list = [
    'chb12_29.edf',
    'chb12_27.edf',
    'chb12_28.edf'
]
df = df[~df['file'].isin(exclude_list)]
df= df.drop(columns='folder')

df.to_csv("all_preprocess_pipeline_seizure.csv", index=False)